In [ ]:
import time
start_time = time.time()

import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    pipeline_device = "mps"
else:
    device = torch.device("cpu")
    pipeline_device = "cpu"

print(f"Selected device: {device}")


In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
classifier = pipeline(
    task="text-classification",
    model=model_name,
    tokenizer=model_name,
    device=pipeline_device,
    truncation=True,
    return_all_scores=False
)
print(f"Loaded pipeline model: {model_name}")


In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print(f"Validation examples: {len(dataset)}")

preview_df = dataset.select(range(min(5, len(dataset)))).to_pandas()
print(preview_df[["sentence1", "sentence2", "label"]].to_string(index=False))


In [ ]:
batch_size = 32
predictions = []
confidence_scores = []
true_labels = []

label2id = {k.upper(): int(v) for k, v in classifier.model.config.label2id.items()}
if not label2id:
    label2id = {"LABEL_0": 0, "LABEL_1": 1}

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    text_pairs = [
        {"text": s1, "text_pair": s2}
        for s1, s2 in zip(batch["sentence1"], batch["sentence2"])
    ]

    outputs = classifier(text_pairs, batch_size=batch_size)
    batch_preds = [label2id[result["label"].upper()] for result in outputs]
    batch_scores = [float(result["score"]) for result in outputs]

    predictions.extend(batch_preds)
    confidence_scores.extend(batch_scores)
    true_labels.extend(batch["label"])

print(f"Completed inference for {len(predictions)} examples.")


In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("Confusion Matrix:")
print(cm)


In [ ]:
results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "split": "validation",
        "num_examples": len(dataset),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "avg_confidence": sum(confidence_scores) / len(confidence_scores),
        "device": str(device)
    }
])

print(results_df.to_string(index=False))


In [ ]:
examples_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
examples_df = examples_df.rename(columns={"label": "true_label"})
examples_df["predicted_label"] = predictions
examples_df["confidence"] = confidence_scores
examples_df["correct"] = examples_df["true_label"] == examples_df["predicted_label"]

print(examples_df.head(10).to_string(index=False))

lowest_conf_df = examples_df.sort_values("confidence", ascending=True).head(10)
print("\nLowest-confidence examples:")
print(lowest_conf_df.to_string(index=False))


In [ ]:
mismatches_df = examples_df[examples_df["true_label"] != examples_df["predicted_label"]].copy()
print(f"Misclassified pairs: {len(mismatches_df)}")
if len(mismatches_df) > 0:
    print(mismatches_df[["sentence1", "sentence2", "true_label", "predicted_label", "confidence"]].head(20).to_string(index=False))


In [ ]:
elapsed_seconds = time.time() - start_time
print(f"Total runtime (seconds): {elapsed_seconds:.2f}")
